# Module 2: Agents vs the ML Pipeline Debugger

Build 4 debugging agents, run a competition across all 3 tasks, and analyse where LLMs fail.

**Time:** ~25 min · **Difficulty:** Intermediate · **GPU:** Not required (CPU inference)

In [ ]:
!pip install -q openenv-core torch numpy openai
import sys, os
sys.path.insert(0, os.path.abspath('ml-pipeline-debugger'))
print('Setup complete!')

In [ ]:
import subprocess, sys, time, os

env_vars = os.environ.copy()
env_vars['PYTHONPATH'] = os.path.abspath('ml-pipeline-debugger')

server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='ml-pipeline-debugger', env=env_vars,
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
time.sleep(4)
import urllib.request
with urllib.request.urlopen('http://localhost:8000/health', timeout=5) as r:
    print('Server:', r.read().decode())

## 1. Import the client

In [ ]:
from client import MLPipelineDebuggerEnv
from models import MLDebugAction
ENV_URL = 'http://localhost:8000'
print('Client imported.')

## 2. Agent 1 — Naive (Baseline)

Returns the broken script unchanged. Scores 0.2–0.3 because the script at least runs.

In [ ]:
class NaiveAgent:
    """Returns the broken script unchanged. Worst possible agent."""
    name = 'Naive'

    def fix(self, obs) -> MLDebugAction:
        return MLDebugAction(
            fixed_code=obs.broken_script,
            explanation='No fix applied.',
            task_id=obs.task_id,
        )

print('NaiveAgent defined.')

## 3. Agent 2 — Keyword Patcher

Uses the `bug_type` field to apply pre-written patches. Fast but brittle — fails on novel bug variants.

In [ ]:
import re

class KeywordPatcherAgent:
    """Patches based on bug_type string — no reasoning, just find-replace."""
    name = 'Keyword Patcher'

    PATCHES = {
        'lr_too_high':          (r'lr=10\.0', 'lr=0.01'),
        'lr_zero':              (r'lr=0\.0\b', 'lr=0.01'),
        'weight_decay_too_high':(r'weight_decay=\d+\.?\d*', 'weight_decay=1e-4'),
        'momentum_negative':    (r'momentum=-\d+\.?\d*', 'momentum=0.9'),
        'epochs_zero':          (r'range\(0\)', 'range(100)'),
        'log_of_zero':          ('np.log(X_np)', 'np.log(np.clip(X_np, 1e-8, None))'),
        'sqrt_negative':        ('np.sqrt(X_np)', 'np.sqrt(np.abs(X_np))'),
        'div_by_zero_std':      ('X_np.std(axis=0)', 'np.where(X_np.std(axis=0)==0, 1, X_np.std(axis=0))'),
        'missing_fillna':       ('return torch.tensor(X_np', 'X_np = np.nan_to_num(X_np, nan=0.0)\n    return torch.tensor(X_np'),
        'inf_from_scale':       (r'\* 1e10', '* 1.0'),
        'zero_weight_init':     ('nn.init.constant_(', '# nn.init.constant_('),
        'frozen_layer':         ('requires_grad = False', 'requires_grad = True'),
        'missing_optimizer_step': ('loss.backward()', 'loss.backward()\n    optimizer.step()'),
        'wrong_activation':     ('lambda t: t * 0', 'torch.relu'),
    }

    def fix(self, obs) -> MLDebugAction:
        code = obs.broken_script
        bug = obs.bug_type
        if bug in self.PATCHES:
            pattern, replacement = self.PATCHES[bug]
            code = re.sub(pattern, replacement, code)
        return MLDebugAction(
            fixed_code=code,
            explanation=f'Applied keyword patch for {bug}',
            task_id=obs.task_id,
        )

print('KeywordPatcherAgent defined.')

## 4. Agent 3 — LLM (GPT-4o)

Sends the broken script and symptom to GPT-4o and returns the fix.

In [ ]:
import os
from openai import OpenAI

SYSTEM_PROMPT = """You are an expert ML engineer who debugs broken PyTorch training scripts.

You receive a broken script, a symptom description, and the bug category.
Return ONLY the complete fixed Python script — no markdown, no explanation.
Every epoch must print: loss:X.XXXXXX
"""

class LLMAgent:
    """Uses GPT-4o to reason about the bug and return a fix."""
    name = 'LLM (GPT-4o)'

    def __init__(self, model='gpt-4o'):
        self.client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))
        self.model = model

    def fix(self, obs) -> MLDebugAction:
        user_msg = f"""BUG TYPE: {obs.bug_type}
SYMPTOM: {obs.task_description}
LOSS CURVE: {obs.loss_curve[:10]}

BROKEN SCRIPT:
{obs.broken_script}

Return the complete fixed script only."""

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': user_msg},
            ],
            temperature=0.0,
            max_tokens=2000,
        )
        fixed = response.choices[0].message.content.strip()
        # Strip markdown fences if model added them
        if fixed.startswith('```'):
            fixed = '\n'.join(l for l in fixed.split('\n') if not l.startswith('```'))
        return MLDebugAction(fixed_code=fixed, explanation='LLM fix', task_id=obs.task_id)

print('LLMAgent defined.')

## 5. Agent 4 — Chain-of-Thought LLM

Forces the model to reason step by step before writing the fix. Typically scores higher on Task 3.

In [ ]:
COT_SYSTEM = """You are an expert ML engineer debugging PyTorch training failures.

STEP 1 — Diagnose: Explain in 2-3 sentences what is wrong and why.
STEP 2 — Fix: Write the complete corrected Python script.

Format your response EXACTLY as:
DIAGNOSIS: <your diagnosis>
FIXED_SCRIPT:
<complete python script that prints loss:X.XXXXXX every epoch>
"""

class ChainOfThoughtAgent:
    """Forces step-by-step reasoning before writing the fix."""
    name = 'Chain-of-Thought LLM'

    def __init__(self, model='gpt-4o'):
        self.client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))
        self.model = model

    def fix(self, obs) -> MLDebugAction:
        user_msg = f"""BUG TYPE: {obs.bug_type}\nSYMPTOM: {obs.task_description}\nLOSS CURVE: {obs.loss_curve[:10]}\n\nBROKEN SCRIPT:\n{obs.broken_script}"""

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {'role': 'system', 'content': COT_SYSTEM},
                {'role': 'user',   'content': user_msg},
            ],
            temperature=0.0,
            max_tokens=2500,
        )
        text = response.choices[0].message.content
        # Extract the script after FIXED_SCRIPT:
        if 'FIXED_SCRIPT:' in text:
            fixed = text.split('FIXED_SCRIPT:', 1)[1].strip()
            fixed = '\n'.join(l for l in fixed.split('\n') if not l.startswith('```'))
        else:
            fixed = obs.broken_script
        diagnosis = text.split('DIAGNOSIS:', 1)[1].split('FIXED_SCRIPT:')[0].strip() if 'DIAGNOSIS:' in text else ''
        return MLDebugAction(fixed_code=fixed, explanation=diagnosis, task_id=obs.task_id)

print('ChainOfThoughtAgent defined.')

## 6. Run the Competition

5 episodes per agent per difficulty level. Requires `OPENAI_API_KEY` for LLM agents.

In [ ]:
import statistics

agents = [NaiveAgent(), KeywordPatcherAgent()]
# Uncomment if OPENAI_API_KEY is set:
# agents += [LLMAgent(), ChainOfThoughtAgent()]

EPISODES_PER_AGENT = 5
all_results = {a.name: [] for a in agents}

with MLPipelineDebuggerEnv(base_url=ENV_URL).sync() as env:
    for agent in agents:
        print(f'\nRunning {agent.name}...')
        for ep in range(EPISODES_PER_AGENT):
            obs = env.reset().observation
            action = agent.fix(obs)
            result = env.step(action)
            all_results[agent.name].append({
                'task': obs.task_id,
                'difficulty': obs.difficulty,
                'bug': obs.bug_type,
                'score': result.reward or 0.0,
            })
            print(f'  ep{ep+1}: {obs.task_id} ({obs.difficulty}) bug={obs.bug_type} score={result.reward:.2f}')

print('\nCompetition complete!')

In [ ]:
# Print summary table
print(f'\n{"Agent":>25} {"Mean Score":>12} {"Min":>6} {"Max":>6}')
print('-' * 55)
for agent_name, results in all_results.items():
    scores = [r['score'] for r in results]
    print(f'{agent_name:>25} {statistics.mean(scores):>12.3f} {min(scores):>6.2f} {max(scores):>6.2f}')

# Score by difficulty
print('\n--- By Difficulty ---')
for agent_name, results in all_results.items():
    by_diff = {}
    for r in results:
        by_diff.setdefault(r['difficulty'], []).append(r['score'])
    print(f'\n{agent_name}:')
    for diff in ['easy', 'medium', 'hard']:
        scores = by_diff.get(diff, [])
        if scores:
            print(f'  {diff:6s}: mean={statistics.mean(scores):.2f}')

In [ ]:
server.terminate(); server.wait(); print('Server stopped.')

## Summary

| Agent | Where it fails |
|-------|----------------|
| Naive | Always — never fixes anything |
| Keyword Patcher | Novel bug variants it hasn't seen |
| LLM (GPT-4o) | Task 3 — silent underfitting with no error message |
| Chain-of-Thought | Still struggles on Task 3 — reasoning helps but isn't enough |

**Next:** Module 3 — Deploying the environment to HuggingFace Spaces.